In [18]:
import os
import random

import numpy as np
import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.experimental.enable_op_determinism()

# Quantization


In [19]:
def quantize(input_tensor, quantize_to_bits=8):
    @tf.custom_gradient
    def straight_through_estimator(input_tensor):
        max_int = (2 ** (quantize_to_bits - 1)) - 1
        max_float = tf.cast(max_int, tf.float32)
        max_val = tf.reduce_max(tf.abs(input_tensor))
        max_val = tf.maximum(max_val, 1e-8)

        scale = max_float / max_val

        quantized = tf.round(input_tensor * scale)
        quantized = tf.clip_by_value(quantized, -max_float, max_float)
        output = quantized / scale

        def grad(upstream, variables=None):
            if variables is not None:
                return upstream, [None] * len(variables)
            return upstream

        return output, grad

    return straight_through_estimator(input_tensor)

In [20]:
from keras.losses import mean_squared_error


def test_quantization():
    weights = tf.Variable([[1.02583, -0.238905], [-0.05612, -1.2983]], dtype=tf.float32)
    print("Original weights:\n", weights.numpy())

    target = tf.constant([[1.0, 1.0], [1.0, -1.0]], dtype=tf.float32)

    with tf.GradientTape() as tape:
        quantized_weights = quantize(weights, quantize_to_bits=4)
        loss = mean_squared_error(target, quantized_weights)
        loss = tf.reduce_mean(loss)

    gradients = tape.gradient(loss, weights)

    print("\nQuantized weights:\n", quantized_weights.numpy())
    print("\nGradients:\n", gradients.numpy())


test_quantization()

Original weights:
 [[ 1.02583  -0.238905]
 [-0.05612  -1.2983  ]]

Quantized weights:
 [[ 1.1128286  -0.18547143]
 [-0.         -1.2983    ]]

Gradients:
 [[ 0.05641431 -0.5927357 ]
 [-0.5        -0.14915001]]


# Model Constructing

## Custom Quantized Dense Layer


In [21]:
from keras.initializers import GlorotUniform
from keras.layers import Layer


class QuantizedDense(Layer):
    def __init__(self, units, quantize_to_bits=8, **kwargs):
        super(QuantizedDense, self).__init__(**kwargs)
        self.units = units
        self.quantize_to_bits = quantize_to_bits

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name="kernel",
            shape=(input_shape[-1], self.units),
            initializer=GlorotUniform(seed=SEED),
            trainable=True,
        )

        self.bias = self.add_weight(
            name="bias", shape=(self.units,), initializer="zeros", trainable=True
        )

    def call(self, inputs):
        quantized_kernel = quantize(self.kernel, quantize_to_bits=self.quantize_to_bits)
        output = tf.matmul(inputs, quantized_kernel)
        output += self.bias
        return output

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.units)

    def get_config(self):
        config = super(QuantizedDense, self).get_config()
        config.update({"units": self.units, "quantize_to_bits": self.quantize_to_bits})
        return config

In [22]:
from keras import Input, Sequential

tf.random.set_seed(42)


def test_quantized_dense():
    model = Sequential([Input(shape=(4,)), QuantizedDense(units=3, quantize_to_bits=4)])

    test_input = tf.random.normal((2, 4))
    test_target = tf.random.normal((2, 3))

    with tf.GradientTape() as tape:
        predictions = model(test_input)
        loss = mean_squared_error(test_target, predictions)
        loss = tf.reduce_mean(loss)

    gradients = tape.gradient(loss, model.trainable_variables)

    print("\nTest X-Y:\n", f"X: {test_input.numpy()}\n", f"Y: {test_target.numpy()}")

    print("\nModel shape:\n", predictions.shape)
    print("\nWeights:\n", model.layers[0].kernel.numpy())
    print("\nGradients", gradients[0].numpy())


test_quantized_dense()


Test X-Y:
 X: [[ 0.3274685 -0.8426258  0.3194337 -1.4075519]
 [-2.3880599 -1.0392479 -0.5573232  0.539707 ]]
 Y: [[ 0.08422458 -0.86090374  0.37812304]
 [-0.00519627 -0.49453196  0.6178192 ]]

Model shape:
 (2, 3)

Weights:
 [[-0.40285552 -0.00293136  0.39895165]
 [-0.8884132  -0.78028464  0.23434246]
 [ 0.4721651  -0.60510516 -0.6874906 ]
 [-0.8282933   0.81940484 -0.49910602]]

Gradients [[-0.6428592  -1.6074047   1.3670007 ]
 [-0.9208706  -0.77439123  0.58027476]
 [ 0.00811617 -0.35666528  0.3226366 ]
 [-0.7225579   0.26192445 -0.32874054]]


## Positional Encoding Layer


In [23]:
class PositionalEncoding(Layer):
    def __init__(self, max_seq_len, dim_model, **kwargs):
        super(PositionalEncoding, self).__init__(**kwargs)
        self.max_seq_len = max_seq_len
        self.dim_model = dim_model

    def build(self, input_shape):
        position = np.arange(self.max_seq_len)[:, np.newaxis]
        i = np.arange(self.dim_model)[np.newaxis, :]

        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(self.dim_model))
        angle_rads = position * angle_rates
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        positional_encoding = angle_rads[np.newaxis, ...]
        self.positional_encoding = tf.cast(positional_encoding, np.float32)

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        return inputs + self.positional_encoding[:, :seq_len, :]

## Multi-Head Attention


In [24]:
class QuantizedMHA(Layer):
    def __init__(self, dim_model, head_num, quantize_to_bits=8, **kwargs):
        super(QuantizedMHA, self).__init__(**kwargs)
        self.dim_model = dim_model
        self.head_num = head_num
        self.quantize_to_bits = quantize_to_bits

        assert dim_model % head_num == 0, "dim_model must be divisible by head_num"

        self.depth = dim_model // head_num
        self.q_dense = QuantizedDense(
            units=dim_model, quantize_to_bits=quantize_to_bits, name="q_layer"
        )
        self.k_dense = QuantizedDense(
            units=dim_model, quantize_to_bits=quantize_to_bits, name="k_layer"
        )
        self.v_dense = QuantizedDense(
            units=dim_model, quantize_to_bits=quantize_to_bits, name="v_layer"
        )
        self.out_dense = QuantizedDense(
            units=dim_model, quantize_to_bits=quantize_to_bits, name="output_layer"
        )

    def split_head(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.head_num, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, q, k, v, mask=None):
        batch_size = tf.shape(q)[0]

        q = self.q_dense(q)
        k = self.k_dense(k)
        v = self.v_dense(v)

        q = self.split_head(q, batch_size)
        k = self.split_head(k, batch_size)
        v = self.split_head(v, batch_size)

        qk = tf.matmul(q, k, transpose_b=True)
        dim_k = tf.cast(self.depth, tf.float32)
        scaled_attention_logits = qk / tf.sqrt(dim_k)

        if mask is not None:
            scaled_attention_logits += mask * -1e9

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)
        output = tf.transpose(output, perm=[0, 2, 1, 3])
        attention = tf.reshape(output, (batch_size, -1, self.dim_model))

        return self.out_dense(attention)

# Forwarding


In [25]:
class ForwardingNetwork(Layer):
    def __init__(self, dim_model, dim_forward, quantize_to_bits=8, **kwargs):
        super(ForwardingNetwork, self).__init__(**kwargs)
        self.dense_1 = QuantizedDense(
            units=dim_forward, quantize_to_bits=quantize_to_bits, name="forward_1"
        )
        self.dense_2 = QuantizedDense(
            units=dim_model, quantize_to_bits=quantize_to_bits, name="forward_2"
        )

    def call(self, x):
        x = self.dense_1(x)
        x = tf.nn.gelu(x)
        return self.dense_2(x)

## Encoder-Decoder

- This one uses standard seq2seq architecture since it achieves higher accuracy for translation tasks


In [43]:
from keras.layers import LayerNormalization


class EncoderLayer(Layer):
    EPSILON = 1e-6

    def __init__(
        self,
        dim_model,
        dim_forward,
        head_num,
        quantize_to_bits=8,
        **kwargs,
    ):
        super(EncoderLayer, self).__init__(**kwargs)
        self.mha = QuantizedMHA(dim_model, head_num, quantize_to_bits)
        self.forwarding = ForwardingNetwork(dim_model, dim_forward, quantize_to_bits)
        self.layer_norm_1 = LayerNormalization(epsilon=self.EPSILON)
        self.layer_norm_2 = LayerNormalization(epsilon=self.EPSILON)

    def call(self, x, training=False, mask=None):
        attention = self.mha(x, x, x, mask)
        output_1 = self.layer_norm_1(x + attention)
        forwarding_output = self.forwarding(output_1)
        return self.layer_norm_2(output_1 + forwarding_output)


class DecoderLayer(Layer):
    EPSILON = 1e-6

    def __init__(
        self,
        dim_model,
        dim_forward,
        head_num,
        quantize_to_bits=8,
        **kwargs,
    ):
        super(DecoderLayer, self).__init__(**kwargs)
        self.mha_1 = QuantizedMHA(dim_model, head_num, quantize_to_bits)
        self.mha_2 = QuantizedMHA(dim_model, head_num, quantize_to_bits)
        self.forwarding = ForwardingNetwork(dim_model, dim_forward, quantize_to_bits)
        self.layer_norm_1 = LayerNormalization(epsilon=self.EPSILON)
        self.layer_norm_2 = LayerNormalization(epsilon=self.EPSILON)
        self.layer_norm_3 = LayerNormalization(epsilon=self.EPSILON)

    def call(
        self, x, encoder_output, training=False, look_ahead_mask=None, padding_mask=None
    ):
        attention_1 = self.mha_1(x, x, x, look_ahead_mask)
        output_1 = self.layer_norm_1(x + attention_1)

        attention_2 = self.mha_2(output_1, encoder_output, encoder_output, padding_mask)
        output_2 = self.layer_norm_2(output_1 + attention_2)

        forwarding_output = self.forwarding(output_2)
        return self.layer_norm_3(output_2 + forwarding_output)

In [44]:
def test_transformer():
    batch_size = 2
    seq_len = 10
    dim_model = 64
    head_num = 4
    dim_forward = 256

    test_x = tf.random.normal((batch_size, seq_len, dim_model))
    test_target = tf.random.normal((batch_size, seq_len, dim_model))
    encoder = EncoderLayer(dim_model, dim_forward, head_num, quantize_to_bits=4)

    with tf.GradientTape() as tape:
        output = encoder(test_x)
        loss = tf.reduce_mean(mean_squared_error(test_target, output))

    gradients = tape.gradient(loss, encoder.trainable_variables)

    print("Encoder shape:\n", output.shape)
    print("\nTrainable weights len:\n", len(encoder.trainable_variables))
    print("\nGradients:\n", gradients[0])


test_transformer()

Encoder shape:
 (2, 10, 64)

Trainable weights len:
 16

Gradients:
 tf.Tensor(
[[-9.03649197e-06 -2.30082253e-04 -1.02281966e-03 ...  2.66312389e-04
   9.77904710e-05  3.54429998e-04]
 [-6.73129398e-04  1.52691937e-05 -3.15960759e-04 ... -5.92891418e-04
  -3.27177113e-04 -1.09773944e-04]
 [-1.48597825e-03  3.58405756e-04 -1.74335856e-03 ... -7.60154799e-04
   4.14119975e-04 -2.50006560e-05]
 ...
 [-1.38758426e-03 -1.99989925e-04 -3.16504668e-03 ...  2.58494867e-04
  -8.12776503e-04 -2.28419434e-04]
 [ 2.85335700e-04 -6.35164906e-06  6.11753203e-05 ...  3.15684825e-04
  -6.17201222e-05  1.11486251e-03]
 [ 2.43104645e-04  8.07243749e-04  2.75305519e-03 ... -2.74152611e-04
  -8.58453335e-04 -3.95301468e-04]], shape=(64, 64), dtype=float32)


## The Model Itself


In [42]:
def generate_padding_mask(seq):
    seq = tf.cast(tf.equal(seq, 0), tf.float32)
    return seq[:, tf.newaxis, tf.newaxis, :]


def generate_look_ahead_mask(seq_len):
    mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
    return mask[tf.newaxis, tf.newaxis, :, :]

In [ ]:
from keras.layers import Embedding


class Encoder(Layer):
    def __init__(
        self,
        layer_num,
        head_num,
        dim_model,
        dim_forward,
        vocab_size,
        max_seq_len,
        quantize_to_bits=8,
        **kwargs,
    ):
        super(Encoder, self).__init__(**kwargs)
        self.dim_model = dim_model
        self.layer_num = layer_num

        self.embedding = Embedding(vocab_size, dim_model)
        self.positional_encoding = PositionalEncoding(max_seq_len, dim_model)

        self.encoder_layers = [
            EncoderLayer(dim_model, dim_forward, head_num, quantize_to_bits)
            for _ in range(layer_num)
        ]

    def call(self, x, training=False, mask=None):
        seq_len = tf.shape(x)[1]

        x = self.embedding(x)
        x *= tf.sqrt(tf.cast(self.dim_model, tf.float32))
        x = self.positional_encoding(x)

        for i in range(self.layer_num):
            x = self.encoder_layers[i](x, training, mask)
        return x


class Decoder(Layer):
    def __init__(
        self,
        layer_num,
        head_num,
        dim_model,
        dim_forward,
        vocal_size,
        max_seq_len,
        quantize_to_bits=8,
        **kwargs,
    ):
        super(Decoder, self).__init__(**kwargs)
        self.dim_model = dim_model
        self.layer_num = layer_num

        self.embedding = Embedding(vocal_size, dim_model)
        self.postional_encoding = PositionalEncoding(max_seq_len, dim_model)

        self.decoder_layers = [
            DecoderLayer(dim_model, dim_forward, head_num, quantize_to_bits)
            for _ in range(layer_num)
        ]

    def call(
        self, x, encoder_output, training=False, look_ahead_mask=None, padding_mask=None
    ):
        seq_len = tf.shape(x)[1]

        x = self.embedding(x)
        x *= tf.sqrt(tf.cast(self.dim_model, tf.float32))
        x = self.postional_encoding(x)

        for i in range(self.layer_num):
            x = self.decoder_layers[i](
                x, encoder_output, training, look_ahead_mask, padding_mask
            )
        return x